In [13]:
import tsl
import torch
import numpy as np
import pandas as pd
import datetime
from tsl.datasets import MetrLA, AirQuality
from einops import rearrange
from torch_geometric.utils.undirected import is_undirected
from tsl.engines import Imputer, Predictor
from torch_geometric.utils.loop import remove_self_loops
from torch_geometric.utils.isolated import contains_isolated_nodes
from tsl.data import SpatioTemporalDataset, ImputationDataset
from torch_geometric.utils import to_dense_adj, to_scipy_sparse_matrix
from tsl.data.datamodule import (SpatioTemporalDataModule, TemporalSplitter)
from tsl.data.preprocessing import StandardScaler
from topomodelx.utils.sparse import from_sparse
from torch_sparse import SparseTensor
from torch_geometric.utils.sparse import to_edge_index
import toponetx as tnx
import networkx as nx
import torch
from torch_cluster import random_walk
import itertools
from utils.random_walk import uniform_random_walk, uniqueness
import torch.nn.functional as F
from tsl.nn.layers.recurrent.base import GraphGRUCellBase
from tsl.nn.blocks.encoders.recurrent.base import RNNBase
from tsl.nn.models import base_model
from tsl.nn import models
from tsl.metrics import numpy as numpy_metrics
from tsl.metrics import torch as torch_metrics
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from tsl.data.preprocessing import StandardScaler, RobustScaler
from pytorch_lightning import Trainer
import math
import gc
import torch.nn as nn

import random
import torch
import numpy as np
import os
from dataset_utils import SDWPE

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.profilers import PyTorchProfiler,AdvancedProfiler
from pytorch_lightning.profilers import AdvancedProfiler

from torch.optim.lr_scheduler import MultiStepLR, CosineAnnealingLR
from pytorch_lightning.loggers import TensorBoardLogger
from utils import MaskedRMSE, TimingCallback

from tsl.nn.models import GRINModel, SPINModel, BiRNNImputerModel, SPINHierarchicalModel
from tsl.ops.imputation import add_missing_values
from tsl.transforms import MaskInput



def seed_everything(seed):
    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True
        torch.set_float32_matmul_precision('medium')  # 'medium' favors performance over precision

        # Enable TF32 format which is optimized for Tensor Cores on Ampere+ GPUs
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    return seed


seed = 42
seed_everything(seed)

42

In [2]:
# dataset = add_missing_values(SDWPE(),
#                             p_fault=0.0015, 
#                             p_noise=0.05,
#                             min_seq=12,
#                             max_seq=12 * 4,
#                             seed=42)

# connectivity = dataset.get_connectivity(threshold=0.1,
#                                         include_self=False,
#                                         # normalize_axis=1,
#                                         force_symmetric=False,
#                                         layout="edge_index")

# covariates = {'u': dataset.datetime_encoded('day').values}

# torch_dataset = ImputationDataset(target=dataset.dataframe(),
#                                       connectivity=connectivity,
#                                      mask=dataset.training_mask,
#                                   eval_mask=dataset.eval_mask,
#                                       # covariates=covariates,
#                                   transform=MaskInput(),
#                                       window=12)
# print(torch_dataset)

ImputationDataset(n_samples=50105, n_nodes=134, n_channels=1)


In [15]:
dataset = add_missing_values(MetrLA(root='./data/metrla'),
                            p_fault=0.0015, 
                            p_noise=0.05,
                            min_seq=12,
                            max_seq=12 * 4,
                            seed=9101112)

connectivity = dataset.get_connectivity(threshold=0.1,
                                        include_self=False,
                                        # normalize_axis=1,
                                        force_symmetric=False,
                                        layout="edge_index")

covariates = {'u': dataset.datetime_encoded('day').values}

torch_dataset = ImputationDataset(target=dataset.dataframe(),
                                      connectivity=connectivity,
                                     mask=dataset.training_mask,
                                  eval_mask=dataset.eval_mask,
                                      covariates=covariates,
                                  transform=MaskInput(),
                                      window=12)
print(torch_dataset)



nodes_in_edges = torch.unique(torch_dataset.edge_index)

total_nodes = torch_dataset.n_nodes

isolated_nodes = total_nodes - len(nodes_in_edges)

isolated_nodes

ImputationDataset(n_samples=34261, n_nodes=207, n_channels=1)


1

In [3]:
# dataset = AirQuality(root='./data/aq', impute_nans=True, small=False)

# splitting = {"val_len": 0.1,
#             "test_len": 0.2}


# connectivity_sparse= {"method": "distance",
#                     "threshold": 0.1,
#                     "include_self": False,
#                     "layout": "edge_index"}

# adj = dataset.get_connectivity(**connectivity_sparse)

# covariates = {'u': dataset.datetime_encoded('day').values}

# torch_dataset = ImputationDataset(target=dataset.dataframe(),
#                                       connectivity=adj,
#                                      mask=dataset.training_mask,
#                                   eval_mask=dataset.eval_mask,
#                                       # covariates=covariates,
#                                   transform=MaskInput(),
#                                       window=12)

# nodes_in_edges = torch.unique(torch_dataset.edge_index)

# total_nodes = torch_dataset.n_nodes

# isolated_nodes = total_nodes - len(nodes_in_edges)

# isolated_nodes

In [16]:
# Normalize data using mean and std computed over time and node dimensions
scalers = {'target': StandardScaler(axis=(0, 1))}

# Split data sequentially:
#   |------------ dataset -----------|
#   |--- train ---|- val -|-- test --|
splitter = TemporalSplitter(val_len=0.1, test_len=0.2)

dm = SpatioTemporalDataModule(
    dataset=torch_dataset,
    scalers=scalers,
    splitter=splitter,
    batch_size=16,
    workers = 4
)

dm.setup()
print(dm)

{Train dataloader: size=24657}
{Validation dataloader: size=2728}
{Test dataloader: size=6852}
{Predict dataloader: None}


In [17]:
loss_fn = torch_metrics.MaskedMAE()
# loss_fn = nn.L1Loss()
log_metrics = {
        'mae': torch_metrics.MaskedMAE(),
        'mre': torch_metrics.MaskedMRE()
        # 'mae_step_2': torch_metrics.MaskedMAE(at=2),
        # 'mae_step_3': torch_metrics.MaskedMAE(at=5),
        # 'mae_step_4': torch_metrics.MaskedMAE(at=11),
        # 'mse_step_2': torch_metrics.MaskedMSE(at=2),
        # 'mse_step_3': torch_metrics.MaskedMSE(at=5),
        # 'mse_step_4': torch_metrics.MaskedMSE(at=11)
    }

# model = GRINModel(input_size =1, hidden_size = 32, 
#                   exog_size = 0, embedding_size = 8,
#                   n_nodes = torch_dataset.n_nodes)

# model = BiRNNImputerModel(input_size =1,exog_size=2, hidden_size =128)


model = SPINModel(input_size = 1, hidden_size = 32, exog_size = 2, n_nodes = torch_dataset.n_nodes, n_layers = 4)

# model = SPINHierarchicalModel(input_size = 1, h_size = 32,z_size = 128, z_heads = 4,exog_size =2,
#                               update_z_cross = False,spatial_aggr='softmax',
#                               n_layers = 5,eta = 3, n_nodes = torch_dataset.n_nodes)

def get_model_log_name(model, torch_dataset):
    class_name = model.__class__.__name__
    directed = str(not is_undirected(torch_dataset.edge_index))
    return f"{class_name}_directed_{directed}"
    

logger = TensorBoardLogger(
        save_dir=f"logs/{dataset.name}",
        name=get_model_log_name(model,torch_dataset)
)

In [18]:
imputer = Imputer(
    model=model,                   # our initialized model
    optim_class=torch.optim.Adam,  # specify optimizer to be used...
    optim_kwargs={'lr': 1e-3,
                  # 'weight_decay':1e-4
                 },    # ...and parameters for its initialization
    loss_fn=loss_fn,               # which loss function to be used
    metrics=log_metrics,                # metrics to be logged during train/val/test
    scale_target = True,
    whiten_prob = 0.05,
    impute_only_missing=False,
    warm_up_steps=0,
    prediction_loss_weight = 1,
    # scheduler_class = CosineAnnealingLR,
    # scheduler_kwargs = {'eta_min':0.0001, 'T_max':200}
)
# 'momentum':0.9,
#                  'nesterov':True


In [19]:
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
experiment_name = f"seed{seed}_{timestamp}"
checkpoint_dir = f'model_checkpoint/{dataset.name}/imputation/{model.__class__.__name__}/{experiment_name}'


checkpoint_callback = ModelCheckpoint(
        dirpath=checkpoint_dir,
        save_top_k=1,
        monitor='val_mae',
        mode='min',
        verbose=True,
        filename='best-{epoch:02d}-{val_mae:.3f}'
)

early_stop_callback = EarlyStopping(
        monitor='val_mae',
        patience=5,
        mode='min',
    min_delta = 0.0001
    )

time_callback = TimingCallback()

trainer = Trainer(
        max_epochs=10,
        limit_train_batches = 150,
       # default_root_dir=cfg.run.dir,
        #logger=exp_logger,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        num_sanity_val_steps=0,
        devices=[0],
        gradient_clip_val=5,
       callbacks=[early_stop_callback,time_callback, checkpoint_callback],
      # default_root_dir="logs",
        # profiler=profiler,
        # precision = '32',
        check_val_every_n_epoch = 3,
        logger=False

    
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [20]:
trainer.fit(imputer, datamodule=dm)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name          | Type             | Params | Mode 
-----------------------------------------------------------
0 | loss_fn       | MaskedMAE        | 0      | train
1 | train_metrics | MetricCollection | 0      | train
2 | val_metrics   | MetricCollection | 0      | train
3 | test_metrics  | MetricCollection | 0      | train
4 | model         | SPINModel        | 71.4 K | train
-----------------------------------------------------------
71.4 K    Trainable params
0         Non-trainable params
71.4 K    Total params
0.286     Total estimated model params size (MB)
220       Modules in train mode
0         Modules in eval mode


Training: |                                                                                                   …

Arguments ['edge_weight'] are filtered out. Only args ['mask', 'x', 'edge_index', 'u'] are forwarded to the model (SPINModel).


Validation: |                                                                                      | 0/? [00:0…

Epoch 2, global step 450: 'val_mae' reached 4.45818 (best 4.45818), saving model to '/netfs/tsp/student/2022/zhu/ST_RUM/model_checkpoint/MissingValuesMetrLA/imputation/SPINModel/seed42_20250715_102147/best-epoch=02-val_mae=4.458.ckpt' as top 1


Validation: |                                                                                      | 0/? [00:0…

Epoch 5, global step 900: 'val_mae' reached 3.66999 (best 3.66999), saving model to '/netfs/tsp/student/2022/zhu/ST_RUM/model_checkpoint/MissingValuesMetrLA/imputation/SPINModel/seed42_20250715_102147/best-epoch=05-val_mae=3.670.ckpt' as top 1


Validation: |                                                                                      | 0/? [00:0…

Epoch 8, global step 1350: 'val_mae' was not in top 1
`Trainer.fit` stopped: `max_epochs=10` reached.



Training Summary: 10 epochs, 304.36s total
Average: 0.0329 epoch/s, 12.28 batch/s


In [21]:
imputer.freeze()

trainer.test(ckpt_path=checkpoint_callback.best_model_path, dataloaders=dm.test_dataloader())

Restoring states from the checkpoint path at /netfs/tsp/student/2022/zhu/ST_RUM/model_checkpoint/MissingValuesMetrLA/imputation/SPINModel/seed42_20250715_102147/best-epoch=05-val_mae=3.670.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at /netfs/tsp/student/2022/zhu/ST_RUM/model_checkpoint/MissingValuesMetrLA/imputation/SPINModel/seed42_20250715_102147/best-epoch=05-val_mae=3.670.ckpt


Testing: |                                                                                         | 0/? [00:0…

Test: 16.26 batch/s, Total: 28.90s


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    0.19973213970661163    │
│         test_mae          │     4.535730838775635     │
│         test_mre          │    0.07855775207281113    │
└───────────────────────────┴───────────────────────────┘

[{'test_mae': 4.535730838775635,
  'test_mre': 0.07855775207281113,
  'test_loss': 0.19973213970661163}]

<table>
  <tr>
    <th>Model</th>
    <th colspan="2" align="center">Metr LA</th>
    <th colspan="2" align="center">AirQuality 36</th>
      <th colspan="2" align="center">AirQuality Full</th>
  </tr>
  <tr>
    <th></th>
    <th>MAE</th>
    <th>MSE</th>
    <th>MAE</th>
    <th>MSE</th>
    <th>MAE</th>
    <th>MSE</th>
  </tr>
  <tr>
    <td>DCRNN Directed</td>
    <td>3.18</td>
    <td>39.67</td>
    <td>-</td>
    <td>-</td>
    <td>-</td>
    <td>-</td>
  </tr>
    <tr>
    <td>DCRNN Undirected</td>
    <td>3.27</td>
    <td>42.04</td>
    <td>31.96</td>
    <td>2593.73</td>
    <td>21.21</td>
    <td>1414.68</td>
  </tr>
  <tr>
    <td>Graph Wavenet Directed</td>
    <td>3.16</td>
    <td>38.88</td>
    <td>-</td>
    <td>-</td>
    <td>-</td>
    <td>-</td>
  </tr>
    <tr>
    <td>Graph Wavenet undirected</td>
    <td>3.24</td>
    <td>41.09</td>
    <td>30.63</td>
    <td>2344.78</td>
    <td>21.07</td>
    <td>1380.89</td>
  </tr>
  <tr>
    <td>Ours</td>
    <td>3.83</td>
    <td>59.78</td>
    <td>33.12</td>
    <td>2699.32</td>
    <td>23.12</td>
    <td>1558.18</td>
  </tr>
</table>

In [6]:
{"SDWPE":{"GRIN":{"#param":"71K", "GPU":"1G", "training":"8.17 batch/s", "testing":"17.54 batch/s"},
          "SPIN":{"#param":"64K", "GPU":"5G", "training":"21.77 batch/s", "testing":"36.82 batch/s"},
          "ModernSASST":{"#param":"100K", "GPU":"2G", "training":"26.05 batch/s", "testing":"43.12 batch/s"}}}


{"AQI":{"GRIN":{"#param":"93K", "GPU":"2G", "training":"7.84 batch/s", "testing":"18.26 batch/s"},
        "SPIN":{"#param":"94K", "GPU":"40G", "training":"4.20 batch/s", "testing":"4.71 batch/s"},
        "ModernSASST":{"#param":"200K", "GPU":"5G", "training":"20.77 batch/s", "testing":"48.99 batch/s"}}}


{"MetrLA":{"GRIN":{"#param":"76K", "GPU":"1G", "training":"8.67 batch/s", "testing":"18.41 batch/s"},
           "SPIN":{"#param":"71K", "GPU":"12G", "training":"12.28 batch/s", "testing":"16.26 batch/s"},
           "ModernSASST":{"#param":"128K", "GPU":"3G", "training":"25.69 batch/s", "testing":"28.23 batch/s"}}}

{'MetrLA': {'GRIN': {'#param': '76K',
   'GPU': '1G',
   'training': '8.67 batch/s',
   'testing': '18.41 batch/s'},
  'SPIN': {'#param': '71K',
   'GPU': '12G',
   'training': '12.28 batch/s',
   'testing': '16.26 batch/s'},
  'Ours': {'#param': '128K',
   'GPU': '3G',
   'training': '25.69 batch/s',
   'testing': '28.23 batch/s'}}}